# General
Project structure:
```
project_root/
├── configs/
│   └── config.py
├── brec/
│   ├── core/
│   │   ├── utils.py
│   │   └── geometry.py
│   ├── data/
│   │   ├── augmentations.py
│   │   ├── cache.py
│   │   └── generators.py
│   ├── models/
│   │   ├── builder.py
│   │   └── losses.py
│   ├── training/
│   │   ├── callbacks.py
│   │   └── trainer.py
│   ├── inference/
│   │   └── reconstructor.py
│   └── evaluation/
│   │   ├── metrics.py
│   │   └── visualizer.py
└── main.py
```

In [ ]:
# %rm -rf *.keras databases

In [ ]:
# %pip install git+https://github.com/ValV/SynthSeg.git
# %pip install -e ./SynthSeg

In [ ]:
from brec.core.env import KAGGLE, configure_xla_paths

if not KAGGLE:
    configure_xla_paths()

In [ ]:
from brec.core.env import PATH_DATA_IXI, PATH_DATA_BRATS, KAGGLE

# Cell 1: Imports & Profiling Utilities
- **Target File:** brec/core/utils.py
- **Content:**
    - PipelineTimer (Context manager for timing)
    - set_global_seeds (Reproducibility)
    - Basic logging configuration

In [ ]:
import numpy as np
import tensorflow as tf
from brec.core.utils import *

In [ ]:
from brec.data.files import *

# Cell 2: The Configuration Class
- **Target File:** configs/config.py
- **Content:**
    - ModelConfig, DataConfig, AugmentationConfig, TrainingConfig dataclasses
    - Config master class
    - Global CFG instantiation (logic for HPO integration)

In [ ]:
from configs.config import CFG, Config, ModelConfig, DataConfig, AugmentationConfig, TrainingConfig, HPOConfig

# Cell 3: Unified Geometry (TensorFlow Native)
- **Target File:** brec/core/geometry.py
- **Content:**
    - GeometryOps class
    - get_smart_crop_coords (Numpy-based)
    - normalize_volume (TF-based)
    - resize_and_pad and inverse_resize_pad (TF-based letterboxing logic)

In [ ]:
from brec.core.geometry import GeometryOps

In [ ]:
from brec.core.geometry import run_geometry_sanity_check

run_geometry_sanity_check(CFG)

In [ ]:
from brec.evaluation.visualizer import analyze_dataset_geometry

# Cell 4: Unified Input Processing
- **Target File:**brec/core/geometry.py
- **Content:**
    - InputProcessor class

In [ ]:
from brec.core.geometry import InputProcessor

# Cell 5: Data Manager with Mask Caching
- **Target File:**brec/data/cache.py
- **Content:**
    - VolumeCache class (Thread-safe LRU cache)
    - DataManager class (Handles IXI/BraTS loading, normalization, and the Hallucination Buffer)

In [ ]:
from brec.data.cache import *

# Cell 6 (Updated): TF-Based Augmentation Logic
- **Target File:**brec/data/augmentations.py (Append to file)
- **Content:**
    - AugmentationLogic class
    - High-level logic: apply_autoregressive_corruption (cascading noise) and apply_tumor_void (mask injection)

In [ ]:
from brec.data.augmentations import AugmentationLogic

# Cell 7: Combined Generators & Mixing Strategy
- **Target File:**brec/data/generators.py
- **Content:**
    - BaseGenerator class
    - IXIDataGenerator class (The main training generator)
    - BraTSDataGenerator class (For mixing real data)
    - create_tf_dataset (Wraps generator in tf.data)
    - get_training_dataset (Mixes the two datasets)

In [ ]:
from brec.data.generators import GeneratorBase

In [ ]:
from brec.data.generators import IXIActiveGenerator, BraTSActiveGenerator

In [ ]:
from brec.data.generators import SequentialValidationGenerator

In [ ]:
from brec.data.generators import HpoTrainSequence, HpoValidSequence

In [ ]:
from brec.data.generators import create_tf_dataset, get_training_dataset

# Cell 8 (Revised): Generic Model Builder

In [ ]:
import tensorflow as tf

- **Target File:**brec/models/layers.py
- **Content:**
    - SPADELayer
    - SPADEResBlock

In [ ]:
from brec.models.layers import *

In [ ]:
# class SpectralNormalization(layers.Wrapper):
#     def __init__(self, layer, iteration=1, **kwargs):
#         super(SpectralNormalization, self).__init__(layer, **kwargs)
#         self.iteration = iteration

#     def build(self, input_shape):
#         if not self.layer.built:
#             self.layer.build(input_shape)
#         if not hasattr(self.layer, 'kernel'):
#             raise ValueError('Layer must have a kernel weight to use SpectralNormalization.')

#         self.w = self.layer.kernel
#         self.w_shape = self.w.shape.as_list()
#         self.u = self.add_weight(shape=(1, self.w_shape[-1]), initializer=tf.initializers.TruncatedNormal(stddev=0.02), trainable=False, name='sn_u', dtype=self.dtype)

#     def call(self, inputs, training=None):
#         # Power iteration
#         # simple implementation for brevity
#         # For full robustness, use tf.keras.layers.SpectralNormalization if available
#         # But here is a simplified version if needed, or rely on standard Conv for now if this is too complex to inject.
#         # Actually, let's try to import the native one first.
#         return self.layer(inputs)

VAE helper classes:

In [ ]:
from brec.models.layers import *

In [ ]:
from brec.models.layers import *

In [ ]:
from brec.models.layers import *

In [ ]:
from brec.models.layers import *

- **Target File:**brec/models/builder.py
- **Content:**
    - ModelBuilder (Factory pattern)
    - _build_unet logic
    - Encoder abstraction (get_efficientnet_encoder)

In [ ]:
from brec.models.builder import ModelBuilder, DiscriminatorBuilder

In [ ]:
from brec.models.builder import ModelBuilder

# Cell 9: Loss Functions
- **Target File:**brec/models/losses.py
- **Content:**
    - CompositeLoss class (Inherits from tf.keras.losses.Loss)
    - Perceptual Loss (VGG/EffNet) lazy initialization logic
    - Spatially Weighted Loss

In [ ]:
from brec.models.losses import *

In [ ]:
from brec.models.losses import *

In [ ]:
from brec.models.losses import *

In [ ]:
from brec.models.losses import *

# Cell 10 (Revised): Training Visualization Callback
- **Target File:**brec/training/callbacks.py
- **Content:**
    - TrainingVisualizer class
    - Logic to plot Input(t-1), GT(t), and Pred(t) every N epochs

In [ ]:
from brec.training.callbacks import TrainingVisualizer

# Cell 11: Updated Trainer with Callbacks
- **Target File:**brec/training/callbacks.py (Part 1) AND brec/training/trainer.py (Part 2)
- **Content:**
    - BufferUpdateCallback -> Goes to brec/training/callbacks.py
    - Trainer class -> Goes to brec/training/trainer.py

In [ ]:
from brec.training.callbacks import BufferUpdateCallback

In [ ]:
from brec.training.callbacks import GeneratorCheckpoint

In [ ]:
from brec.training.trainer import SPADEGANTrainer

In [ ]:
from brec.training.trainer import Trainer

# Cell 12: Metrics & Visualization
- **Target File:** brec/evaluation/metrics.py
- **Content:**
    - Evaluator class
    - calculate_region_metrics (SSIM/PSNR on masks)
    - Basic ortho-slice plotting helpers

In [ ]:
from brec.evaluation.metrics import *

In [ ]:
from brec.evaluation.metrics import *

In [ ]:
from brec.evaluation.metrics import *

In [ ]:
from brec.evaluation.metrics import *

In [ ]:
from brec.evaluation.metrics import *

# Cell 13: The Unified Reconstructor
- **Target File:** brec/inference/reconstructor.py
- **Content:**
    - VolumeReconstructor class
    - Inference logic: _predict_slice, autoregressive_restore, bidirectional_restore

In [ ]:
from brec.inference.reconstructor import VolumeReconstructor

# Cell 14 (Unused): Execution Check
- **Target:** Notebook cell only (or tests/test_inference.py)
- **Content:**
    - Sanity checks for the Reconstructor on dummy data

# Cell 15: The Visualization Suite
- **Target File:**brec/evaluation/visualizer.py
- **Content:**
    - VisualizationSuite class
    - All plotting logic: plot_augmentations, plot_training_history, plot_hallucination_buffer, analyze_best_worst, plot_autoregressive_performance, plot_bidirectional

In [ ]:
from brec.evaluation.visualizer import VolumeDashboard

In [ ]:
from brec.evaluation.visualizer import VisualizationSuite

In [ ]:
from brec.evaluation.visualizer import VisualizationSuite

# Cell 16: The Main Execution Block (`main.py`)
- **Target File:** main.py
- **Content:**
    - run_pipeline() function
    - Orchestration of Config -> Data -> Model -> Trainer -> Visualization

In [ ]:
# %rm -rfv *.json databases

In [ ]:
from main import run_pipeline, run_eda_mode, run_ablation_study, run_ablation_study_, get_ablation_configs, calculate_step_metrics, calc_focal_frequency_error, calc_gradient_sharpness_error

In [ ]:
from main import run_pipeline, run_ablation_study, run_ablation_study_

* **Target File**: cvpr_plots.py

In [ ]:
import cvpr_plots

In [ ]:
from cvpr_plots import generate_figure_1_drift_curve, generate_figure_2_evolution_matrix, generate_figure_3_pe_proof, generate_figure_4_anatomical_dsc, generate_figure_5_frechet_distances, generate_figure_6_pareto, generate_supp_bidirectional, generate_supp_ablation_masked_ar, generate_supp_scaling_failure, run_cvpr_rendering, PALETTE, LABELS

# Cell 17: The Optuna Interface

In [ ]:
from brec.hpo.engine import compute_autoregressive_score, HPMEngine

In [ ]:
from brec.hpo.engine import compute_autoregressive_score, HPMEngine

## Static version

In [ ]:
from brec.hpo.engine import HPMEngine

In [ ]:
from brec.hpo.engine import HPMEngine

# Cell 18: The Merging Utility

In [ ]:
from brec.hpo.merge import merge_hpo_databases

In [ ]:
from brec.hpo.merge import merge_hpo_databases

# Cell 19: HPO Visualization Analysis

In [ ]:
from brec.hpo.analysis import analyze_hpo_results

In [ ]:
from brec.hpo.analysis import analyze_hpo_results

# Cell 20: Extra evaluation metrics

In [ ]:
from brec.evaluation.synthseg import SynthSegEvaluator

In [ ]:
from brec.evaluation.frechet import FrechetEvaluator

In [ ]:
from brec.evaluation.fastsurfer import FastSurferEvaluator

In [ ]:
from brec.evaluation.monai_sota import MonaiSotaEvaluator

# Run Pipeline

In [ ]:
# %mv -v kaggle-brats-results.zip kaggle-brats-results-$(date +%Y%m%d-%H%M%S).zip

In [ ]:
# !zip -r -9 kaggle-brats-results.zip ablation_results paper_figures memory_telemetry.csv

In [ ]:
# from IPython.display import FileLink


# FileLink('kaggle-brats-results.zip')

In [ ]:
# %rm -rfv ablation_results/nifti
# %rm -rf ablation_results/fastsurfer_masks
# %rm -rf ablation_results paper_figures

In [ ]:
from main import main

# Valid modes: 'train', 'hpo', 'eda', 'ablation', 'sota', 'dsc', 'frechet', 'cvpr', 'all'
mode = 'ablation'
main(mode=mode)

In [ ]:
%rm -rf FastSurfer SynthSeg
%ls -allah *

In [ ]:
%cat /kaggle/working/ablation_results/inference_performance.csv

In [ ]:
from brec.evaluation.visualizer import display_random_dashboards

display_random_dashboards()

In [ ]:
# %%time
# import sys
# import pkg_resources
# import subprocess
# from collections import defaultdict


# # Get a list of all imported modules
# imported_modules = {
#     name: module
#     for name, module in sys.modules.items()
#     if module and getattr(module, '__file__', None)
# }

# # Get the version and dependencies of each imported module
# package_info = defaultdict(dict)

# for name in imported_modules:
#     try:
#         # Skip non-package modules
#         if name.startswith('_') or name in sys.builtin_module_names:
#             continue

#         dist = pkg_resources.get_distribution(name)
#         version = dist.version
#         package_info[name]['version'] = version

#         # Get dependencies using pip
#         result = subprocess.run(
#             ['pip', 'show', name], capture_output=True, text=True
#         )
#         dependencies = []
#         for line in result.stdout.split('\n'):
#             if line.startswith('Requires: '):
#                 dependencies = [
#                     d.strip()
#                     for d in line.split('Requires: ')[1].split(',')
#                     if d.strip()
#                 ]
#                 break

#         package_info[name]['dependencies'] = dependencies
#     except (pkg_resources.DistributionNotFound, subprocess.CalledProcessError):
#         pass

# # Print the package information
# for name, info in package_info.items():
#     print(f"Package: {name}")
#     print(f"Version: {info.get('version', 'N/A')}")
#     print(f"Dependencies: {', '.join(info.get('dependencies', []))}")
#     print()